# JGI Integration Workflow

Interactive version of `python -m tools.cli run`. Every cell calls the same stage function as the command line (`tools/pipeline.py`), so results are identical; here you also see figures inline. Outputs go to hash-tagged folders: `Dataset_Processing--<hash>/Analysis--<hash>/`.

In [ ]:
import pandas as pd
import tools.objects as objs
import tools.pipeline as pl
from tools.runs import list_runs

pd.options.display.max_columns = None

CONFIG_DIR = "/home/jovyan/work/input_data/config"   # project.yml, data_processing.yml, analysis.yml
RESULTS_DIR = "/home/jovyan/work/output_data"       # project.results_path
NOTEBOOK_PATH = "workflow.ipynb"                     # saved with the run record

In [ ]:
# Optional: runs that already exist (resume one by passing its hashes below)
list_runs(RESULTS_DIR)

## Project setup
To resume or re-run a saved configuration, set both hashes and point `config_dir` at `<RESULTS_DIR>/<project_name>/configs`.

In [ ]:
project = objs.Project(config_dir=CONFIG_DIR, data_processing_hash=None, analysis_hash=None, overwrite=False)
analysis = pl.build_analysis(project)   # one Dataset object per entry in project.dataset_list

## Link samples between datasets

In [ ]:
pl.stage_link(analysis)

## Data processing

In [ ]:
# Validate/detect each dataset's input scale, then filter rare raw features (presence and magnitude)
pl.stage_filter(analysis)

In [ ]:
# Sample-normalize (raw input only), impute mx/px zeros, log/VST-transform
pl.stage_normalize(analysis)

In [ ]:
# Remove features not impacted by experimentation
pl.stage_devariance(analysis)

In [ ]:
# Remove features with outlier replicates
pl.stage_replicability(analysis)

In [ ]:
# Scale each dataset at sample (paired) or condition (unpaired) resolution
pl.stage_scale(analysis)

## Data type integration

In [ ]:
pl.stage_integrate(analysis)

## Feature selection

In [ ]:
pl.stage_select_features(analysis)

## Knowledge-driven and data-driven groups of features

In [ ]:
# Knowledge-driven: full annotation scope (KEGG, ModelSEED, KO, COG, EC, BRITE, compound classes...)
pl.stage_annotate(analysis)

In [ ]:
# Data-driven: group features by co-variation (method from analysis.yml feature_grouping)
pl.stage_group_features(analysis)

## Data-driven vs knowledge-driven groups

In [ ]:
# Enrichment of every annotation type in every group (+ heatmaps per type in enrichment/)
enrichment = pl.stage_enrich(analysis)

## Explorer bundle and run record

In [ ]:
# Precompute statistics, embeddings and structure for the interactive explorer (explorer/ folder)
pl.stage_build_explorer(analysis)
project.save_run_record(notebook_path=NOTEBOOK_PATH)

## Explore the results
Run the next cell to start the interactive explorer for this run, then open **http://localhost:8050** in your browser (the container must publish port 8050; see the setup guide).

The explorer runs in the background: it keeps running if you restart or close this notebook. Run `stop_explorer()` to stop it. Re-running the cell restarts it.

To explore a different run, find its hashes with `list_runs(RESULTS_DIR)` and pass that run's `Analysis--<hash>` folder to `start_explorer`. From a terminal, the equivalent is `jgi-integrate explore --project-dir <results>/<project> -d <data hash> -a <analysis hash>`.

In [ ]:
from tools.explorer_launch import start_explorer, stop_explorer

start_explorer(project.analysis_dir)

In [ ]:
# stop_explorer()